# PolyglotSound : worker GPU (Colab / Kaggle)

Exécuter les cellules **de haut en bas**. Résultat : une URL `https://xxxx.trycloudflare.com`
à copier dans `WORKER_URL` du backend.

* Colab : *Exécution > Modifier le type d'exécution > GPU T4*.
* Kaggle : *Settings > Accelerator > GPU T4 x2* (pas P100) et *Internet : On*.

Routes : `/health /separate /transcribe /align /analyze /translate /tts /convert /sing_auto /watermark/embed /watermark/detect`
(+ `/warmup` et `/unload` hors contrat). Spécification : `docs/CONTRACT.md`.

## 1. Vérification du GPU

In [5]:
!nvidia-smi
import torch, sys
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA dispo :", torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("GPU :", torch.cuda.get_device_name(0), f"| VRAM libre {free/1e9:.1f} / {total/1e9:.1f} Go")
else:
    print("ATTENTION : pas de GPU. Activez un GPU T4 avant de continuer.")

Sat Nov  7 09:22:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             33W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Configuration

* `REPO_URL` : dépôt GitHub contenant `worker/` (laisser vide pour téléverser les fichiers à la main).
* `WORKER_TOKEN` : secret partagé avec le backend (variable `WORKER_TOKEN` côté backend aussi). Vide = pas d'auth.

In [6]:
import os, secrets
REPO_URL = "https://github.com/KOBA2008/PolygotSound-.git"  # dépôt de l'équipe
BRANCH = "dev"        # "main" une fois dev fusionné
WORKER_TOKEN = secrets.token_urlsafe(16)  # aléatoire à chaque session, affiché avec WORKER_URL (cellule tunnel)
PORT = 9000
SEEDVC_STEPS = 25     # 30 = meilleure qualité mais risque de dépasser la coupure ~100 s du tunnel

IS_KAGGLE = os.path.isdir("/kaggle/working")
BASE = "/kaggle/working" if IS_KAGGLE else "/content"
WORKER_DIR = f"{BASE}/polyglotsound/worker"
SEEDVC_DIR = f"{BASE}/seed-vc"
os.makedirs(WORKER_DIR, exist_ok=True)
print("Plateforme :", "Kaggle" if IS_KAGGLE else "Colab", "| dossier worker :", WORKER_DIR)

Plateforme : Colab | dossier worker : /content/polyglotsound/worker


## 3. Récupération du code du worker

Option A (GitHub) si `REPO_URL` est rempli. Option B sinon : téléverser `server.py`, `seedvc_runner.py`,
`requirements.txt`, `test_client.py` (sur Kaggle : les ajouter comme *Dataset* puis les copier, voir README).

In [7]:
import os, shutil, subprocess, glob
if REPO_URL:
    repo_dir = f"{BASE}/polyglotsound"
    if os.path.isdir(f"{repo_dir}/.git"):
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
    else:
        if os.path.isdir(repo_dir):
            shutil.rmtree(repo_dir)
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, repo_dir], check=True)
else:
    # Kaggle : fichiers ajoutés comme Dataset (/kaggle/input/...)
    for f in glob.glob("/kaggle/input/**/worker/*", recursive=True) + glob.glob("/kaggle/input/*/*.py") + glob.glob("/kaggle/input/*/requirements.txt"):
        if os.path.basename(f) in ("server.py", "seedvc_runner.py", "requirements.txt", "test_client.py"):
            shutil.copy(f, WORKER_DIR)
    missing = [f for f in ("server.py", "seedvc_runner.py", "requirements.txt") if not os.path.exists(f"{WORKER_DIR}/{f}")]
    if missing:
        try:
            from google.colab import files
            print("Téléversez :", ", ".join(missing), "(+ test_client.py en option)")
            up = files.upload()
            for name, data in up.items():
                with open(f"{WORKER_DIR}/{os.path.basename(name)}", "wb") as fh:
                    fh.write(data)
        except ImportError:
            raise SystemExit(f"Fichiers manquants dans {WORKER_DIR} : {missing}")
print(sorted(os.listdir(WORKER_DIR)))
assert os.path.exists(f"{WORKER_DIR}/server.py"), "server.py manquant"

['PolyglotSound_Worker.ipynb', 'README.md', 'requirements.txt', 'seedvc_runner.py', 'server.py', 'test_client.py']


## 4. Installation des dépendances (3 à 6 min)

On ne réinstalle **pas** torch. torchaudio est installé seulement s'il manque, à la version exacte de torch.

In [4]:
import subprocess, sys, importlib, re
import torch

def pip(*args):
    print("pip", " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        raise RuntimeError("échec pip " + " ".join(args))

# torchaudio assorti à torch (Colab l'a normalement déjà)
try:
    import torchaudio
    print("torchaudio présent :", torchaudio.__version__)
except Exception:
    m = re.match(r"(\d+\.\d+\.\d+)(?:\+(cu\d+|cpu))?", torch.__version__)
    ver, flavor = m.group(1), (m.group(2) or "cu126")
    pip(f"torchaudio=={ver}", "--index-url", f"https://download.pytorch.org/whl/{flavor}")

pip("-r", f"{WORKER_DIR}/requirements.txt")
# descript-audiotools épingle protobuf<3.20 : installé sans dépendances (celles utiles sont dans requirements.txt)
pip("--no-deps", "descript-audiotools==0.7.2", "descript-audio-codec==1.0.0")

for mod in ["fastapi", "uvicorn", "multipart", "soundfile", "librosa", "transformers", "demucs", "audioseal",
            "uroman", "torchcrepe", "pyworld", "torchaudio", "dac", "munch", "einops"]:
    try:
        m = importlib.import_module(mod)
        print(f"OK  {mod:13} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"ERR {mod:13} {e}")
import torchaudio
print("torch", torch.__version__, "| torchaudio", torchaudio.__version__)
try:
    from torchaudio.pipelines import MMS_FA
    print("MMS_FA disponible")
except Exception as e:
    print("MMS_FA indisponible :", e)

torchaudio présent : 2.11.0+cu130
pip -r /content/polyglotsound/worker/requirements.txt
pip --no-deps descript-audiotools==0.7.2 descript-audio-codec==1.0.0
OK  fastapi       0.141.1
OK  uvicorn       0.54.0
OK  multipart     0.0.33
OK  soundfile     0.14.0
OK  librosa       0.11.0
OK  transformers  5.22.1
OK  demucs        4.1.0
OK  audioseal     0.2.1
OK  uroman        


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


OK  torchcrepe    
OK  pyworld       0.3.5
OK  torchaudio    2.11.0+cu130


/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


OK  dac           1.0.0
OK  munch         4.0.0
OK  einops        0.8.2
torch 2.14.1+cu130 | torchaudio 2.11.0+cu130
MMS_FA disponible


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 5. Clonage de Seed-VC (poids téléchargés au premier appel de /convert, ~1.5 Go)

In [8]:
import os, subprocess
if not os.path.isfile(f"{SEEDVC_DIR}/inference.py"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Plachtaa/seed-vc.git", SEEDVC_DIR], check=True)
print(subprocess.run(["git", "-C", SEEDVC_DIR, "log", "-1", "--format=%h %cd %s"], capture_output=True, text=True).stdout)
print("inference.py :", os.path.isfile(f"{SEEDVC_DIR}/inference.py"))

51383ef Sun Apr 20 13:27:04 2025 +0800 fixe dataloader build bug when fine-tuning v1 model

inference.py : True


## 6. Lancement du worker (uvicorn en arrière-plan, port 9000)

In [ ]:
import os, subprocess, sys, time, requests
env = dict(os.environ, SEEDVC_DIR=SEEDVC_DIR, WORKER_TOKEN=WORKER_TOKEN, PYTHONUNBUFFERED="1",
           TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1", SEEDVC_STEPS=str(SEEDVC_STEPS))
LOG = f"{BASE}/worker.log"
try:
    worker_proc.terminate(); worker_proc.wait(10)
except Exception:
    pass
subprocess.run("fuser -k %d/tcp" % PORT, shell=True, capture_output=True)
subprocess.run("pkill -f seedvc_runner.py", shell=True, capture_output=True)  # ancien démon Seed-VC
worker_proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", str(PORT)],
                               cwd=WORKER_DIR, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
HEADERS = {"X-Worker-Token": WORKER_TOKEN} if WORKER_TOKEN else {}
LOCAL = f"http://127.0.0.1:{PORT}"
for _ in range(60):
    try:
        r = requests.get(LOCAL + "/health", headers=HEADERS, timeout=5)
        if r.ok:
            print("Worker prêt :", r.json()); break
    except Exception:
        time.sleep(1)
else:
    print(open(LOG).read()[-3000:])
    raise RuntimeError("Le worker ne répond pas, voir le log ci-dessus")

## 7. Warmup (optionnel mais conseillé, 5 à 12 min la première fois)

Télécharge et charge chaque modèle en l'appelant une fois sur un son synthétique.
Mettre `WARMUP = "translate,tts"` pour ne précharger qu'une partie.

In [ ]:
import requests, json, time
WARMUP = "all"   # ou "separate,transcribe,align,analyze,translate,tts,watermark,seedvc"
t0 = time.time()
r = requests.post(LOCAL + "/warmup", data={"models": WARMUP}, headers=HEADERS, timeout=3600)
print(f"{time.time()-t0:.0f} s")
print(json.dumps(r.json(), indent=1, ensure_ascii=False))

## 8. Tunnel public cloudflared (URL trycloudflare)

In [ ]:
import os, re, subprocess, time, urllib.request
from IPython.display import HTML, display
CF = f"{BASE}/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
CF_LOG = f"{BASE}/cloudflared.log"
try:
    tunnel_proc.terminate()
except Exception:
    pass
tunnel_proc = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                               stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT)
WORKER_URL = None
for _ in range(90):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(CF_LOG).read())
    if m:
        WORKER_URL = m.group(0); break
if not WORKER_URL:
    print(open(CF_LOG).read()[-3000:])
    raise RuntimeError("URL trycloudflare introuvable")
print("WORKER_URL =", WORKER_URL)
print("WORKER_TOKEN =", WORKER_TOKEN)
display(HTML(f'''<div style="padding:18px;border-radius:12px;background:#0f0b1f;color:#a3e635;font:700 26px monospace">
WORKER_URL={WORKER_URL}<br>WORKER_TOKEN={WORKER_TOKEN}</div><p style="font-size:15px">À mettre dans l'environnement du backend
(<code>backend/.env</code>). L'URL et le jeton changent à chaque relance du tunnel.</p>'''))

## 9. Tests : /health puis chaque route sur un son de test (via l'URL publique)

In [ ]:
import requests, subprocess, sys, os
print(requests.get(WORKER_URL + "/health", headers=HEADERS, timeout=30).json())
cmd = [sys.executable, f"{WORKER_DIR}/test_client.py", WORKER_URL, "--out", f"{BASE}/worker_test_out", "--lang", "fon", "--target", "eng"]
if WORKER_TOKEN:
    cmd += ["--token", WORKER_TOKEN]
# Pour tester avec un vrai refrain : cmd += ["--audio", "/content/refrain.wav"]
if os.path.exists(f"{WORKER_DIR}/test_client.py"):
    subprocess.run(cmd)
else:
    print("test_client.py absent : seul /health a été testé")

### Écouter les sorties du test

In [ ]:
import glob
from IPython.display import Audio, display
for f in sorted(glob.glob(f"{BASE}/worker_test_out/*.wav")):
    print(os.path.basename(f)); display(Audio(f))

## 10. Journal du worker / maintien en vie

In [ ]:
print(open(LOG).read()[-5000:])
if os.path.exists("/tmp/seedvc_daemon.log"):
    print("--- Seed-VC ---"); print(open("/tmp/seedvc_daemon.log").read()[-3000:])

In [ ]:
# Garde la session active et affiche périodiquement l'état (interrompre avec le bouton stop).
import time, requests
while True:
    try:
        h = requests.get(LOCAL + "/health", headers=HEADERS, timeout=10).json()
        print(time.strftime("%H:%M:%S"), WORKER_URL, h.get("models_loaded"), h.get("vram"))
    except Exception as e:
        print("health KO :", e)
    if worker_proc.poll() is not None:
        print("Le worker s'est arrêté !"); print(open(LOG).read()[-3000:]); break
    time.sleep(120)